# Proyecto de Sistemas Inteligentes — Solicitudes Universitarias con LangChain

**Asignatura:** Sistemas Inteligentes 1 — Universidad de Caldas

En este notebook muestro cómo funciona nuestro proyecto paso a paso. La idea del
sistema es recibir solicitudes de la universidad (por ejemplo "no puedo entrar a
Moodle") y que la inteligencia artificial las **clasifique**, les ponga una
**prioridad** y diga **quién debería atenderlas**.

Este notebook es **independiente**: tiene todo el código adentro y no necesita
ninguna otra carpeta del proyecto. Solo hace falta internet (para hablar con la IA)
y las librerías de LangChain, que se instalan en la primera celda.

Lo voy probando por partes:

1. Instalamos las librerías y ponemos la configuración (las claves).
2. Probamos que la conexión con la IA (Azure OpenAI) funcione.
3. Definimos las categorías, prioridades y el prompt.
4. Hacemos que la IA responda siempre con un formato ordenado.
5. Creamos la herramienta que asigna el responsable (con datos guardados aquí mismo).
6. Juntamos todo en una sola cadena de LangChain y la probamos con varios ejemplos.
7. Por último, probamos el algoritmo A\* que reparte las solicitudes entre los funcionarios.

## 0. Instalar las librerías

Si es la primera vez que corro el notebook en esta máquina, instalo lo que necesita.
Si ya las tengo instaladas, esta celda no hace daño (solo confirma que están).

In [ ]:
%pip install -q langchain-core langchain-openai pydantic

## 1. Configuración (las claves de la IA)

Aquí pongo los datos de conexión a Azure OpenAI. Normalmente esto iría en un archivo
`.env` aparte, pero como necesito que el notebook funcione solo, lo dejo escrito
directamente.

> Nota: la clave queda visible en el notebook. Si lo voy a compartir, lo ideal sería
> borrar o cambiar la clave después.

In [ ]:
# ---- Configuración de Azure OpenAI ----
AZURE_OPENAI_API_KEY = "AGREGAR LUEGO"
AZURE_OPENAI_ENDPOINT = "AGREGAR LUEGO"
AZURE_OPENAI_DEPLOYMENT = "AGREGAR LUEGO"
AZURE_OPENAI_API_VERSION ="AGREGAR LUEGO"
LLM_TEMPERATURE = 0.1  # baja, para que las respuestas sean estables

print("Endpoint   :", AZURE_OPENAI_ENDPOINT)
print("Deployment :", AZURE_OPENAI_DEPLOYMENT)
print("API key    :", AZURE_OPENAI_API_KEY[:6] + "…" + AZURE_OPENAI_API_KEY[-4:])

## 2. Probar la conexión con la IA (Azure OpenAI)

Creo el modelo de IA con las claves de arriba y le mando una pregunta muy corta para
ver si responde. Así confirmo que la conexión está bien antes de seguir.

In [ ]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    azure_deployment=AZURE_OPENAI_DEPLOYMENT,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=LLM_TEMPERATURE,
)

print("Tipo de modelo:", type(llm).__name__)
print("Deployment    :", llm.deployment_name)
print("Temperatura   :", llm.temperature)

# Llamada de prueba para verificar que el endpoint responde.
respuesta = llm.invoke("Responde solo con la palabra: OK")
print("\nRespuesta de la IA:", respuesta.content)

## 3. Categorías, prioridades y el prompt

Aquí defino las categorías y prioridades posibles (como listas fijas) y el prompt, que
son las instrucciones que le doy a la IA. En el prompt le explico bien qué significa
cada categoría y cada prioridad para que no se invente cosas.

In [ ]:
from enum import Enum
from langchain_core.prompts import ChatPromptTemplate


class Categoria(str, Enum):
    """Categorías posibles de una solicitud universitaria."""
    ACADEMICA = "Académica"
    FINANCIERA = "Financiera"
    TECNOLOGICA = "Tecnológica"
    ADMINISTRATIVA = "Administrativa"


class Prioridad(str, Enum):
    """Niveles de prioridad con que se atiende una solicitud."""
    ALTA = "Alta"
    MEDIA = "Media"
    BAJA = "Baja"


SISTEMA_CLASIFICADOR = """\
Eres un asistente experto de la Universidad de Caldas encargado de clasificar
y priorizar solicitudes universitarias. Tu trabajo es analizar el asunto y la
descripción de cada solicitud y devolver una clasificación estructurada.

Debes asignar EXACTAMENTE una CATEGORÍA entre estas:
- Académica: matrículas, notas, cursos, docentes, horarios, exámenes, grados.
- Financiera: pagos, becas, créditos, facturación, devoluciones, descuentos.
- Tecnológica: plataformas (Moodle, correo, wifi), accesos, software, equipos.
- Administrativa: certificados, trámites, constancias, documentos, atención general.

Debes asignar EXACTAMENTE una PRIORIDAD entre estas:
- Alta: bloquea por completo al usuario, tiene fecha límite inminente, o afecta
  a muchas personas (caída de plataforma, vencimiento de pago, cierre de matrícula).
- Media: genera molestia o demora pero existe alternativa o el plazo no es urgente.
- Baja: consulta general, información o solicitud sin impacto inmediato.

Reglas:
1. Basa tu decisión únicamente en el contenido del asunto y la descripción.
2. Sé consistente: ante el mismo tipo de solicitud, clasifica siempre igual.
3. En 'razonamiento' explica de forma breve (1-2 frases) por qué elegiste esa
   categoría y prioridad.
4. Responde siempre en español.
"""

HUMANO_CLASIFICADOR = """\
Clasifica la siguiente solicitud universitaria.

ASUNTO: {asunto}
DESCRIPCIÓN: {descripcion}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SISTEMA_CLASIFICADOR),
        ("human", HUMANO_CLASIFICADOR),
    ]
)

# Veamos cómo queda el prompt ya armado con un ejemplo:
for m in prompt.format_messages(
    asunto="No puedo acceder a Moodle",
    descripcion="Error 500 al iniciar sesión y tengo examen mañana.",
):
    print(f"===== {m.type.upper()} =====")
    print(m.content)
    print()

## 4. Hacer que la IA responda con un formato ordenado

Un problema típico es que la IA conteste con texto suelto y después toque adivinar qué
parte es la categoría y qué parte la prioridad. Para evitar eso defino un modelo con
Pydantic (`ClasificacionResult`) y uso `with_structured_output`, que obliga a la IA a
devolver siempre los mismos campos: `categoria`, `prioridad` y `razonamiento`.

In [ ]:
from pydantic import BaseModel, Field


class ClasificacionResult(BaseModel):
    """Lo que la IA debe devolver siempre, con esta forma exacta."""
    categoria: Categoria = Field(..., description="Categoría asignada a la solicitud.")
    prioridad: Prioridad = Field(..., description="Prioridad asignada a la solicitud.")
    razonamiento: str = Field(
        ..., description="Justificación breve de por qué se asignó esa categoría y prioridad."
    )


llm_estructurado = llm.with_structured_output(ClasificacionResult)

cadena_parcial = prompt | llm_estructurado  # prompt -> IA ordenada

resultado = cadena_parcial.invoke(
    {
        "asunto": "Error en el cobro de matrícula",
        "descripcion": "Me cobraron dos veces la matrícula y necesito la "
        "devolución antes del cierre financiero del viernes.",
    }
)

print("Tipo devuelto :", type(resultado).__name__)
print("Categoría     :", resultado.categoria.value)
print("Prioridad     :", resultado.prioridad.value)
print("Razonamiento  :", resultado.razonamiento)

## 5. La herramienta que asigna el responsable

En el proyecto original esta herramienta consulta una base de datos en Azure. Para que
el notebook funcione solo, aquí guardo el mismo catálogo de funcionarios como una
lista de Python (son los mismos datos que estaban en la base de datos).

Luego creo una herramienta de LangChain (`asignar_responsable`) que, según la
categoría, busca en esa lista qué área se encarga.

In [ ]:
from langchain_core.tools import tool

# Catálogo de funcionarios (mismos datos que la tabla 'funcionarios' de la BD).
FUNCIONARIOS = [
    {"id": 1, "nombre": "Coordinación Académica",     "categoria": "Académica",      "correo": "academica@ucaldas.edu.co"},
    {"id": 2, "nombre": "Departamento Financiero",    "categoria": "Financiera",     "correo": "financiera@ucaldas.edu.co"},
    {"id": 3, "nombre": "Soporte de Tecnología (TI)", "categoria": "Tecnológica",    "correo": "soporte.ti@ucaldas.edu.co"},
    {"id": 4, "nombre": "Secretaría Administrativa",  "categoria": "Administrativa", "correo": "administrativa@ucaldas.edu.co"},
]

RESPONSABLE_POR_DEFECTO = "Secretaría Administrativa (sin asignación específica)"


@tool
def asignar_responsable(categoria: str) -> str:
    """Asigna el área responsable de atender una solicitud según su categoría.

    Busca en el catálogo de funcionarios y devuelve el nombre del área junto con
    su correo. Si la categoría no existe, devuelve un responsable por defecto.
    """
    for f in FUNCIONARIOS:
        if f["categoria"] == categoria:
            return f"{f['nombre']} ({f['correo']})"
    return RESPONSABLE_POR_DEFECTO


# Primero veo el catálogo:
print(f"Funcionarios disponibles: {len(FUNCIONARIOS)}\n")
for f in FUNCIONARIOS:
    print(f"  [{f['id']}] {f['nombre']:30s} | {f['categoria']:14s} | {f['correo']}")

# Y ahora pruebo la herramienta con cada categoría:
print("\nPrueba de la herramienta:")
for categoria in ["Tecnológica", "Financiera", "Académica", "Administrativa"]:
    responsable = asignar_responsable.invoke({"categoria": categoria})
    print(f"  {categoria:14s} -> {responsable}")

## 6. Juntar todo en una sola cadena

Esta es la parte principal. Uno las tres cosas anteriores con el operador `|` de
LangChain (esto se llama LCEL): primero el prompt, después la IA, y al final la
herramienta que asigna el responsable. La función `clasificar_solicitud` hace todo de
una sola vez: clasifica, le pone prioridad y dice quién la atiende.

Lo pruebo con tres ejemplos distintos:

In [ ]:
from langchain_core.runnables import RunnableLambda


def _agregar_responsable(clasificacion: ClasificacionResult) -> dict:
    """Toma lo que respondió la IA y le añade el responsable usando la herramienta."""
    responsable = asignar_responsable.invoke({"categoria": clasificacion.categoria.value})
    return {
        "categoria": clasificacion.categoria,
        "prioridad": clasificacion.prioridad,
        "razonamiento": clasificacion.razonamiento,
        "responsable": responsable,
    }


# Cadena completa: prompt -> IA ordenada -> agregar responsable
cadena = prompt | llm_estructurado | RunnableLambda(_agregar_responsable)


def clasificar_solicitud(asunto: str, descripcion: str) -> dict:
    """Clasifica una solicitud y le asigna responsable, todo en un paso."""
    return cadena.invoke({"asunto": asunto, "descripcion": descripcion})


EJEMPLOS = [
    {
        "asunto": "No puedo acceder a Moodle",
        "descripcion": "Desde ayer me sale error 500 al iniciar sesión en la "
        "plataforma y tengo un examen mañana.",
    },
    {
        "asunto": "Consulta sobre fechas de grado",
        "descripcion": "Quisiera saber cuándo abren las inscripciones para la "
        "próxima ceremonia de grados.",
    },
    {
        "asunto": "Error en el cobro de matrícula",
        "descripcion": "Me cobraron dos veces el valor de la matrícula y necesito "
        "la devolución antes del cierre financiero del viernes.",
    },
]

for ejemplo in EJEMPLOS:
    r = clasificar_solicitud(**ejemplo)
    print("=" * 72)
    print("ASUNTO       :", ejemplo["asunto"])
    print("Categoría    :", r["categoria"].value)
    print("Prioridad    :", r["prioridad"].value)
    print("Responsable  :", r["responsable"])
    print("Razonamiento :", r["razonamiento"])

## 7. Repartir las solicitudes con el algoritmo A\*

Además de la IA, el proyecto tiene un algoritmo de búsqueda **A\***. En vez de asignar
una solicitud a la vez, agarra un grupo de solicitudes pendientes y las reparte entre
los funcionarios tratando de que el "costo" total sea el más bajo posible. El costo
tiene en cuenta qué tan ocupado está cada funcionario, cuánto se demora y si la
solicitud es urgente.

Abajo está el algoritmo completo (es el mismo del proyecto, copiado aquí para que el
notebook sea independiente).

In [ ]:
import heapq
import itertools
import time
from dataclasses import dataclass, field

# Pesos y umbrales de la función de costo
PESO_CARGA = 1.0
PESO_TIEMPO_RESPUESTA = 1.0
PENALIZACION_URGENCIA = 2.0
UMBRAL_CARGA_ALTA = 4
UMBRAL_TIEMPO_LENTO = 2.0


@dataclass(frozen=True)
class SolicitudPendiente:
    id: int
    categoria: str
    prioridad: str


@dataclass(frozen=True)
class FuncionarioCandidato:
    id: int
    nombre: str
    especialidad: str
    carga_actual: int
    tiempo_promedio_respuesta: float
    disponibilidad_horas: int


@dataclass
class AsignacionResultado:
    solicitud_id: int
    funcionario_id: int
    funcionario_nombre: str
    costo: float


@dataclass
class ResultadoAStar:
    asignaciones: list
    costo_total: float
    nodos_explorados: int
    tiempo_ejecucion_ms: float
    sin_solucion: list = field(default_factory=list)


def _normalizar(valor, maximo):
    if maximo <= 0:
        return 0.0
    return min(max(valor / maximo, 0.0), 1.0)


def _costo_asignacion(solicitud, funcionario, max_carga, max_tiempo):
    """Costo de asignar una solicitud a un funcionario.

    Mezcla qué tan cargado está y qué tan lento es. Si la solicitud es urgente
    y el funcionario está saturado o es lento, le sumo una penalización para
    que A* prefiera no mandarle urgencias.
    """
    costo = PESO_CARGA * _normalizar(funcionario.carga_actual, max_carga)
    costo += PESO_TIEMPO_RESPUESTA * _normalizar(
        funcionario.tiempo_promedio_respuesta, max_tiempo
    )
    es_urgente = solicitud.prioridad == Prioridad.ALTA.value
    esta_saturado = funcionario.carga_actual >= UMBRAL_CARGA_ALTA
    es_lento = funcionario.tiempo_promedio_respuesta >= UMBRAL_TIEMPO_LENTO
    if es_urgente and (esta_saturado or es_lento):
        costo += PENALIZACION_URGENCIA
    return costo


def _candidatos_por_solicitud(solicitudes, funcionarios):
    """Para cada solicitud, los funcionarios cuya especialidad coincide."""
    return [
        [j for j, f in enumerate(funcionarios) if f.especialidad == s.categoria]
        for s in solicitudes
    ]


def _heuristica(indice, solicitudes, candidatos, costos_por_par):
    """Estimación optimista del costo que falta (suma del candidato más barato)."""
    total = 0.0
    for i in range(indice, len(solicitudes)):
        idxs = candidatos[i]
        if not idxs:
            continue
        total += min(costos_por_par[(i, j)] for j in idxs)
    return total


def ejecutar_astar(solicitudes, funcionarios):
    """Reparte el lote de solicitudes entre los funcionarios con búsqueda A*."""
    inicio = time.perf_counter()

    if not solicitudes:
        return ResultadoAStar([], 0.0, 0, (time.perf_counter() - inicio) * 1000)

    candidatos = _candidatos_por_solicitud(solicitudes, funcionarios)
    max_carga = max((f.carga_actual for f in funcionarios), default=0) or 1
    max_tiempo = max((f.tiempo_promedio_respuesta for f in funcionarios), default=0) or 1

    costos_por_par = {}
    for i, s in enumerate(solicitudes):
        for j in candidatos[i]:
            costos_por_par[(i, j)] = _costo_asignacion(
                s, funcionarios[j], max_carga, max_tiempo
            )

    sin_candidatos = [s.id for i, s in enumerate(solicitudes) if not candidatos[i]]
    capacidad_inicial = tuple(f.disponibilidad_horas for f in funcionarios)

    contador = itertools.count()
    nodos_explorados = 0
    mejor_costo_visto = {}

    h_inicial = _heuristica(0, solicitudes, candidatos, costos_por_par)
    heap = [(h_inicial, 0.0, next(contador), 0, capacidad_inicial, ())]

    while heap:
        _f, g, _orden, indice, capacidades, asignaciones_parciales = heapq.heappop(heap)
        nodos_explorados += 1

        clave_estado = (indice, capacidades)
        if mejor_costo_visto.get(clave_estado, float("inf")) <= g:
            continue
        mejor_costo_visto[clave_estado] = g

        if indice == len(solicitudes):
            asignaciones = [
                AsignacionResultado(
                    solicitud_id=solicitudes[i].id,
                    funcionario_id=funcionarios[j].id,
                    funcionario_nombre=funcionarios[j].nombre,
                    costo=costo,
                )
                for (i, j, costo) in asignaciones_parciales
            ]
            return ResultadoAStar(
                asignaciones, g, nodos_explorados,
                (time.perf_counter() - inicio) * 1000, sin_candidatos,
            )

        idxs_candidatos = candidatos[indice]
        if not idxs_candidatos:
            # Nadie puede atender esta solicitud: avanzo sin asignarla.
            nuevo_indice = indice + 1
            nuevo_h = _heuristica(nuevo_indice, solicitudes, candidatos, costos_por_par)
            heapq.heappush(heap, (
                g + nuevo_h, g, next(contador), nuevo_indice,
                capacidades, asignaciones_parciales,
            ))
            continue

        for j in idxs_candidatos:
            if capacidades[j] <= 0:
                continue
            nuevas_capacidades = list(capacidades)
            nuevas_capacidades[j] -= 1
            nuevas_capacidades = tuple(nuevas_capacidades)

            costo_accion = costos_por_par[(indice, j)]
            nuevo_g = g + costo_accion
            nuevo_indice = indice + 1
            nuevo_h = _heuristica(nuevo_indice, solicitudes, candidatos, costos_por_par)
            heapq.heappush(heap, (
                nuevo_g + nuevo_h, nuevo_g, next(contador), nuevo_indice,
                nuevas_capacidades,
                asignaciones_parciales + ((indice, j, costo_accion),),
            ))

    # Se acabaron las opciones sin asignar todo: no hay capacidad suficiente.
    return ResultadoAStar(
        [], 0.0, nodos_explorados,
        (time.perf_counter() - inicio) * 1000, [s.id for s in solicitudes],
    )


print("Algoritmo A* listo.")

Ahora pruebo A\* con un ejemplo inventado: pongo dos funcionarios de Tecnología, uno
rápido y otro ya saturado, para ver cómo reparte las solicitudes.

In [ ]:
# Lote de solicitudes pendientes (categoría + prioridad ya clasificadas).
lote = [
    SolicitudPendiente(id=101, categoria="Tecnológica", prioridad="Alta"),
    SolicitudPendiente(id=102, categoria="Tecnológica", prioridad="Media"),
    SolicitudPendiente(id=103, categoria="Tecnológica", prioridad="Alta"),
    SolicitudPendiente(id=104, categoria="Financiera", prioridad="Baja"),
    SolicitudPendiente(id=105, categoria="Académica", prioridad="Media"),
]

# Dos funcionarios de Tecnología con distinta carga/velocidad: A* debe repartir.
candidatos = [
    FuncionarioCandidato(1, "TI - Ana (rápida)", "Tecnológica",
                         carga_actual=1, tiempo_promedio_respuesta=1, disponibilidad_horas=2),
    FuncionarioCandidato(2, "TI - Beto (saturado)", "Tecnológica",
                         carga_actual=5, tiempo_promedio_respuesta=3, disponibilidad_horas=2),
    FuncionarioCandidato(3, "Financiera - Caro", "Financiera",
                         carga_actual=0, tiempo_promedio_respuesta=1, disponibilidad_horas=8),
    FuncionarioCandidato(4, "Académica - Dani", "Académica",
                         carga_actual=2, tiempo_promedio_respuesta=1, disponibilidad_horas=8),
]

resultado = ejecutar_astar(lote, candidatos)

print("ASIGNACIONES ÓPTIMAS (A*)")
print("-" * 60)
for a in resultado.asignaciones:
    print(f"  Solicitud #{a.solicitud_id}  ->  {a.funcionario_nombre:24s}  (costo {a.costo:.3f})")

print("-" * 60)
print(f"Costo total        : {resultado.costo_total:.3f}")
print(f"Nodos explorados   : {resultado.nodos_explorados}")
print(f"Tiempo de ejecución: {resultado.tiempo_ejecucion_ms:.2f} ms")
print(f"Sin solución (ids) : {resultado.sin_solucion}")

### ¿Qué pasó aquí?

Como se ve en el resultado, las solicitudes urgentes (prioridad **Alta**) se las dio a
*Ana*, que está más libre y es más rápida, en vez de a *Beto*, que está saturado.
Esto es justo lo que queríamos: que las cosas urgentes vayan con quien las puede
atender más pronto. Y como Ana solo tenía 2 horas libres, A\* también repartió algo a
Beto para no dejar solicitudes sin atender. Así se nota que la parte de IA
(clasificar y priorizar) se conecta con la parte de optimización (repartir bien el
trabajo).

## Para terminar

En resumen, en este notebook fuimos probando cada parte del proyecto, todo dentro del
mismo archivo:

- Pusimos las claves y probamos que la IA de Azure OpenAI respondiera (§1 y §2).
- Definimos las categorías, prioridades y el prompt con las instrucciones (§3).
- Logramos que la IA respondiera siempre ordenada (§4).
- Creamos la herramienta que asigna el responsable (§5).
- Juntamos todo en una sola cadena de LangChain (§6).
- Y por último repartimos las solicitudes con el algoritmo A\* (§7).

Con esto se ve que el sistema cumple lo que pedía el proyecto: usa un LLM, un prompt,
una herramienta propia, las une con LangChain (LCEL) y además le suma una parte de
optimización con A\*.